### 🛠️ Step 1: Environment Setup
We begin by importing **PyTorch** (`torch`), which serves as our core deep learning library, and the standard math module for auxiliary operations.

In [2]:
import torch
import math

### 🍃 Step 2: Creating a Leaf Node with Autograd Tracking
To calculate gradients with respect to an input tensor, we must explicitly instruct PyTorch to track its operations.

* **`requires_grad=True`**: This flag marks the tensor `x` as a variable rather than a constant. PyTorch will now track all subsequent operations involving `x` dynamically.
* **Leaf Node**: In the dynamic computation graph constructed by PyTorch, `x` acts as a leaf node because it was created directly by the user rather than as a result of a tracked operation.

In [3]:
x = torch.tensor(5.0, requires_grad=True)

### 📈 Step 3: Forward Pass & Defining the Objective Function
Here, we define a scalar function $f(x)$ using standard mathematical operations:
$$f(x) = x^2 + \frac{1}{x} + 5$$

Because $f(x)$ is computed using the tracked variable `x`, PyTorch automatically builds a directed acyclic graph (DAG) representing this computation.

In [22]:
f = x ** 2 + 1/x + 5

### 🔍 Step 4: Inspecting the Forward Pass Result
We can evaluate the output tensor $f$. Notice that along with the calculated value of `30.2000`, the tensor holds a dynamic reference to the operation that finalized it.

In [23]:
f

tensor(30.2000, grad_fn=<AddBackward0>)

### 🕸️ Step 5: Understanding the Computation Graph (`grad_fn`)
Every non-leaf tensor produced by a tracked operation contains a special attribute:

* **`grad_fn`**: This attribute points to a backward function (in this case, `<AddBackward0>`) that knows how to backpropagate gradients through the final operation that created the tensor. This chain of `grad_fn` objects defines the computation graph.

In [24]:
f.grad_fn

### ⏪ Step 6: Triggering Backpropagation
To compute the derivatives of $f$ with respect to our tracked parameters, we call the backward function:

* **`f.backward()`**: This initiates reverse-mode automatic differentiation. PyTorch traverses the computation graph backward from $f$ down to all leaf nodes (like `x`), computing and accumulating gradients along the way.

In [6]:
f.backward()

### 🎯 Step 7: Inspecting the Accumulated Gradients
Once the backward pass is complete, the calculated derivative $\frac{df}{dx}$ is stored directly within the leaf tensor's `.grad` attribute.

Mathematically, for $f(x) = x^2 + x^{-1} + 5$:
$$\frac{df}{dx} = 2x - \frac{1}{x^2}$$

Evaluating this at $x = 5.0$:
$$\frac{df}{dx}\Big|_{x=5} = 2(5) - \frac{1}{25} = 10 - 0.04 = 9.96$$

*(Note: Any tiny numerical differences in printed output are due to floating-point precision constraints).*

In [7]:
x.grad

tensor(10.)

---

## 🔄 Section 2: Gradient Accumulation and In-Place Updates

### ➕ Step 8: Defining Multiple Computation Paths
In PyTorch, a single leaf tensor can contribute to multiple different outputs. Here, we define two separate mathematical pathways, $f$ and $g$, derived from the same base variable `a`:
* $f(a) = a^2$
* $g(a) = 2a^3$

In [47]:
a = torch.tensor(5.0, requires_grad=True)
f = a**2
g = 2 * a**3

### 🔋 Step 9: Running Multiple Backward Passes
When we call `.backward()` on multiple tensors sequentially, PyTorch's default behavior is to **accumulate (add)** the gradients into the leaf tensor's `.grad` attribute rather than overwriting them.

* $\frac{df}{da} = 2a$
* $\frac{dg}{da} = 6a^2$

In [48]:
f.backward()
g.backward()

### 🎯 Step 10: Inspecting the Accumulated Gradients on Leaf Node `a`
Let's evaluate the final accumulated gradient in `a.grad` at $a = 5.0$:
$$\text{Accumulated Gradient} = \frac{df}{da} + \frac{dg}{da} = 2a + 6a^2$$
$$\text{At } a=5.0: \quad 2(5) + 6(25) = 10 + 150 = 160$$

In [49]:
a.grad

tensor(160.)

### 🛡️ Step 11: Modifying Tensors In-Place using `torch.no_grad()`
When performing gradient descent updates, we modify our parameters directly. However, we must prevent PyTorch from tracking these update steps themselves in the computation graph.

* **`with torch.no_grad()`**: Disables gradient tracking temporarily so that the parameter update operation is not recorded, avoiding infinite loops or building unnecessary graphs during optimization.

In [51]:
learning_rate = 0.1
with torch.no_grad():
    a -= learning_rate * a.grad  # gradient descent step

### 🔗 Step 12: Updating Parameters using `detach()`
An alternative way to safely update or use a tensor without tracking its history is to create a detached view of it.

* **`a.detach()`**: Creates a new tensor `a_detached` that shares the same storage as `a` but does not require gradients and is not part of the active computation graph.

In [53]:
a_detached = a.detach()
a_detached -= learning_rate * a.grad;

---

## 🔄 Section 3: Training Loops & Computational Pitfalls

### 🔄 Step 13: The Optimization Loop & Gradient Zeroing
When training machine learning models over multiple iterations, we repeatedly compute gradients, update weights, and repeat. However, because of PyTorch's default **gradient accumulation** behavior, we must manually clear the tracking buffer on every step.

* **`x.grad.zero_()`**: Crucially resets the accumulated gradients of `x` to zero at the end of each iteration. Without this line, gradients from previous steps would add up continually, leading to incorrect updates and exploding gradients.

In [21]:
learning_rate = 0.001
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(2800):
    f = x ** 2  # forward pass
    f.backward()  # backward pass
    with torch.no_grad():
        x -= learning_rate * x.grad  # gradient descent step

    x.grad.zero_()  # reset the gradients

In [22]:
x

tensor(0.0184, requires_grad=True)

### ⚠️ Step 14: In-Place Operations and Backpropagation Failure
While in-place operations (like `+=`, `*=`) are memory-efficient, they can break automatic differentiation.

* **The Graph Dependency**: During backpropagation, PyTorch needs the unmodified intermediate values (like the output of `t.exp()`) to compute the derivatives of downstream nodes.
* **The RuntimeError**: Performing an in-place modification (`z += 1`) overwrites the values that PyTorch's dynamic graph needs for backward pass calculation, causing a runtime crash.

In [ ]:
t = torch.tensor(2.0, requires_grad=True)
z = t.exp()  # this is an intermediate result
z += 1  # this is an in-place operation
z.backward()  #  RuntimeError!
